In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/abalone/train.csv')

# Display the first few rows of the dataset
print(train_data.head())

# Check the shape of the dataset
print(train_data.shape)

# Check for missing values
print(train_data.isnull().sum())

# Check the data types of each column
print(train_data.dtypes)

# Separate numerical and categorical columns
numerical_cols = train_data.select_dtypes(include=[np.number]).columns
categorical_cols = train_data.select_dtypes(include=['object']).columns

print("Numerical Columns:", numerical_cols)
print("Categorical Columns:", categorical_cols)

# Visualize the distribution of numerical features
for col in numerical_cols:
    plt.figure(figsize=(6, 4))
    sns.histplot(train_data[col], bins=30, kde=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.show()

# Visualize the distribution of categorical features
for col in categorical_cols:
    plt.figure(figsize=(6, 4))
    sns.countplot(x=train_data[col])
    plt.title(f'Count of {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.show()

# Check for correlation between numerical features
plt.figure(figsize=(10, 8))
correlation_matrix = train_data[numerical_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Numerical Features')
plt.show()


      id Sex  Length  ...  Whole weight.2  Shell weight  Rings
0  43718   M   0.650  ...          0.3020         0.360     10
1  45247   F   0.670  ...          0.3455         0.385     11
2  71393   M   0.605  ...          0.2470         0.285      9
3  51688   M   0.680  ...          0.3060         0.440     11
4  40681   I   0.295  ...          0.0190         0.038      5

[5 rows x 10 columns]
(72492, 10)
id                0
Sex               0
Length            0
Diameter          0
Height            0
Whole weight      0
Whole weight.1    0
Whole weight.2    0
Shell weight      0
Rings             0
dtype: int64
id                  int64
Sex                object
Length            float64
Diameter          float64
Height            float64
Whole weight      float64
Whole weight.1    float64
Whole weight.2    float64
Shell weight      float64
Rings               int64
dtype: object
Numerical Columns: Index(['id', 'Length', 'Diameter', 'Height', 'Whole weight', 'Whole weight.1',
  

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_data)
print("column_info")
print(column_info)


2025-09-15 00:21:08.182 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['Sex'], 'Numeric': ['id', 'Length', 'Diameter', 'Height', 'Whole weight', 'Whole weight.1', 'Whole weight.2', 'Shell weight', 'Rings'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, StandardScale

# Load the training and test datasets
train_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/abalone/train.csv')
test_data = pd.read_csv(r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/abalone/test.csv')

# Separate features and target
X_train = train_data.drop(columns=['Rings'])
y_train = train_data['Rings']
X_test = test_data.drop(columns=['Rings'])
y_test = test_data['Rings']

# Handle missing values
fill_missing = FillMissingValue(features=['Length', 'Diameter', 'Height', 'Whole weight', 'Whole weight.1', 'Whole weight.2', 'Shell weight'], strategy='mean')
X_train = fill_missing.fit_transform(X_train)
X_test = fill_missing.transform(X_test)

# Encode categorical variables
label_encode = LabelEncode(features=['Sex'])
X_train = label_encode.fit_transform(X_train)
X_test = label_encode.transform(X_test)

# Normalize numerical features
standard_scale = StandardScale(features=['Length', 'Diameter', 'Height', 'Whole weight', 'Whole weight.1', 'Whole weight.2', 'Shell weight'])
X_train = standard_scale.fit_transform(X_train)
X_test = standard_scale.transform(X_test)

# Display the preprocessed data
print(X_train.head())
print(X_test.head())


      id  Sex    Length  ...  Whole weight.1  Whole weight.2  Shell weight
0  43718    2  1.125265  ...        1.809351        1.316374      1.030331
1  45247    0  1.294627  ...        1.449565        1.748091      1.222442
2  71393    2  0.744199  ...        1.065303        0.770526      0.453999
3  51688    2  1.379309  ...        1.709003        1.356072      1.645085
4  40681    1 -1.880923  ...       -1.497260       -1.492265     -1.444054

[5 rows x 9 columns]
      id  Sex    Length  ...  Whole weight.1  Whole weight.2  Shell weight
0   3502    1 -0.229637  ...       -0.608807       -0.316209     -0.544976
1  76031    1 -0.822406  ...       -0.873140       -0.891831     -0.948408
2  49473    2  0.320792  ...        0.052026       -0.117719      0.069778
3  16126    0  0.151429  ...       -0.442375        0.006338     -0.276021
4  65405    0  0.744199  ...        0.424050        0.512488      0.646110

[5 rows x 9 columns]


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(X_train)
print("column_info")
print(column_info)


column_info
{'Category': [], 'Numeric': ['id', 'Sex', 'Length', 'Diameter', 'Height', 'Whole weight', 'Whole weight.1', 'Whole weight.2', 'Shell weight'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
from xgboost import XGBRegressor

# Assuming X_train, X_test, y_train, y_test are already defined from previous tasks

# Initialize and train the XGBoost model
model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train, early_stopping_rounds=50, eval_set=[(X_test, y_test)], verbose=100)

# Predict on the test set
y_pred = model.predict(X_test)

# Calculate RMSE
rmse = np.sqrt(mean_squared_log_error(y_test, y_pred))
print(f'RMSE: {rmse}')


TypeError: XGBModel.fit() got an unexpected keyword argument 'early_stopping_rounds'